# Full-Population Ingestion

Builds the canonical master table for the 200k Representative pipeline.

**Analytical universe:** English-only notes, at least 3 English notes per tweet,
48-hour rating eligibility downstream, and at least 3 eligible votes per note
before clustering.

**Inputs (place in `raw/`):**
- `notes-*.tsv`
- `ratings-*.tsv`
- `noteStatusHistory-00000.tsv`
- FastText language-id model, usually `lid.176.ftz`

**Output:** `data/master_full.parquet`.

With `TEST_MODE=1`, input is limited to one ratings shard and 200,000 rows and
all output is isolated under `.artifacts/smoke/`.


In [ ]:
import sys
from pathlib import Path
import glob
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.io import (
    filter_notes_by_fasttext_language,
    get_config_float,
    get_config_int,
    get_config_value,
    get_data_root,
    get_test_mode,
)

TEST_MODE = get_test_mode()
DATA_ROOT = get_data_root()
MAX_SHARDS = 1 if TEST_MODE else None
MAX_ROWS_PER_SHARD = 200_000 if TEST_MODE else None
FASTTEXT_LID_MODEL = Path(get_config_value('FASTTEXT_LID_MODEL', 'raw/lid.176.ftz')).expanduser()
if not FASTTEXT_LID_MODEL.is_absolute():
    FASTTEXT_LID_MODEL = PROJECT_ROOT / FASTTEXT_LID_MODEL
FASTTEXT_BINARY = Path(get_config_value('FASTTEXT_BINARY', 'raw/fasttext')).expanduser()
if not FASTTEXT_BINARY.is_absolute():
    FASTTEXT_BINARY = PROJECT_ROOT / FASTTEXT_BINARY
LANGUAGE_FILTER = get_config_value('LANGUAGE_FILTER', 'en')
LANGUAGE_MIN_CONFIDENCE = get_config_float('LANGUAGE_MIN_CONFIDENCE', 0.5)
LANGUAGE_BATCH_SIZE = get_config_int('LANGUAGE_BATCH_SIZE', 10000)

print(f'=== TEST_MODE = {TEST_MODE}'
      f' ({"smoke test (small sample)" if TEST_MODE else "FULL DATA"}) ===')
print(f'    DATA_ROOT          = {DATA_ROOT}')
print(f'    MAX_SHARDS         = {MAX_SHARDS}')
print(f'    MAX_ROWS_PER_SHARD = {MAX_ROWS_PER_SHARD}')
print(f'    FASTTEXT_LID_MODEL = {FASTTEXT_LID_MODEL}')
print(f'    FASTTEXT_BINARY    = {FASTTEXT_BINARY}')
print(f'    LANGUAGE_FILTER    = {LANGUAGE_FILTER}')
print(f'    LANG_MIN_CONF      = {LANGUAGE_MIN_CONFIDENCE}')

RAW = PROJECT_ROOT / 'raw'
OUT = DATA_ROOT
SHARD_DIR = OUT / f'_shards_lang_{LANGUAGE_FILTER}_v1'
OUT.mkdir(parents=True, exist_ok=True)
SHARD_DIR.mkdir(parents=True, exist_ok=True)

notes_paths = sorted(glob.glob(str(RAW / 'notes-*.tsv')))
assert notes_paths, f'No notes-*.tsv files found in {RAW}'
assert (RAW / 'noteStatusHistory-00000.tsv').exists(), f'missing {RAW}/noteStatusHistory-00000.tsv'
rating_paths = sorted(glob.glob(str(RAW / 'ratings-*.tsv')))
if MAX_SHARDS is not None:
    rating_paths = rating_paths[:MAX_SHARDS]
print(f'rating shards selected: {len(rating_paths)}')

## 1) Load notes, keep English notes, then define valid tweets (`>= 3 English notes per tweet`)

In [ ]:
notes = pd.concat(
    [
        pd.read_csv(
            p, sep='\t',
            usecols=['noteId', 'tweetId', 'createdAtMillis', 'summary', 'classification'],
            low_memory=False,
        )
        for p in notes_paths
    ],
    ignore_index=True,
)
print(f'notes shards loaded:          {len(notes_paths)}')
print(f'notes before language filter: {len(notes):,}')

notes = filter_notes_by_fasttext_language(
    notes,
    model_path=FASTTEXT_LID_MODEL,
    language=LANGUAGE_FILTER,
    min_confidence=LANGUAGE_MIN_CONFIDENCE,
    batch_size=LANGUAGE_BATCH_SIZE,
    fasttext_binary=FASTTEXT_BINARY or None,
)
print(f'notes after language filter:  {len(notes):,}')
print(notes['noteLanguage'].value_counts(dropna=False).head())

counts = notes.groupby('tweetId')['noteId'].nunique()
valid_tweets = set(counts[counts >= 3].index)
notes = notes[notes['tweetId'].isin(valid_tweets)].copy()
valid_notes = set(notes['noteId'])
print(f'valid tweets (>=3 English notes): {len(valid_tweets):,}')
print(f'valid English notes:              {len(valid_notes):,}')

## 2) Filter each rating shard, write to per-shard parquet

Per-shard files are written under `data/_shards_lang_en_v1/`, or under the
equivalent `.artifacts/smoke/` path in smoke mode. Each expected shard is
overwritten on every ingest run.


In [ ]:
for fp in rating_paths:
    shard_name = Path(fp).stem + '.parquet'
    shard_out = SHARD_DIR / shard_name
    if shard_out.exists():
        shard_out.unlink()
        print(f'overwrite stale shard: {shard_name}')
    r = pd.read_csv(
        fp,
        sep='\t',
        usecols=['noteId', 'raterParticipantId', 'helpfulnessLevel', 'createdAtMillis'],
        low_memory=False,
        nrows=MAX_ROWS_PER_SHARD,
    )
    r = r[r['noteId'].isin(valid_notes)]
    r.to_parquet(shard_out, index=False)
    print(f'wrote {shard_name}: {len(r):,} rows')

## 3) Concatenate filtered ratings, then merge with note metadata + status history

In [ ]:
# In TEST_MODE, only consume the shards we actually wrote in this run
# (so a previous full-mode shards directory doesn't leak in).
expected_shards = [SHARD_DIR / (Path(fp).stem + '.parquet') for fp in rating_paths]
ratings = pd.concat(
    [pd.read_parquet(p) for p in expected_shards if p.exists()],
    ignore_index=True,
)
print(f'total filtered ratings: {len(ratings):,}')

status = pd.read_csv(
    RAW / 'noteStatusHistory-00000.tsv',
    sep='\t',
    usecols=['noteId', 'currentStatus', 'timestampMillisOfFirstNonNMRStatus'],
    low_memory=False,
)
status = status[status['noteId'].isin(valid_notes)]
print(f'status rows: {len(status):,}')

df = (
    ratings
    .merge(notes.rename(columns={'createdAtMillis': 'noteCreatedAtMillis'}), on='noteId', how='left')
    .merge(status.rename(columns={'timestampMillisOfFirstNonNMRStatus': 'firstNonNMRMillis'}), on='noteId', how='left')
)
out_path = OUT / 'master_full.parquet'
df.to_parquet(out_path, index=False)
print(f'wrote {out_path}')
print(f'rows={len(df):,}')
print(f'unique tweets: {df["tweetId"].nunique():,}')
print(f'unique notes:  {df["noteId"].nunique():,}')
print(f'unique raters: {df["raterParticipantId"].nunique():,}')
df.head()